## Agenda

1. How the DATA step processes: compilation and execution
2. Declarative vs executable statements
3. The program data vector (PDV)
4. Implicit vs explicit `OUTPUT`
5. The `RETAIN` statement

## More on the DATA step

Let's talk more about what exactly happens when we run a DATA step.

We will use a small food survey. Each student rated three restaurants on food quality and service.

```{=html}
<table class="mini"><tr><th>Student</th><th>Restaurant</th><th>FoodQuality</th><th>Service</th></tr>
<tr><td>Vince</td><td>Kings</td><td>7</td><td>8</td></tr>
<tr><td>Vince</td><td>Jamaican Grill</td><td>8</td><td>6.5</td></tr>
<tr><td>Vince</td><td>Fuji</td><td>9</td><td>6</td></tr>
<tr><td>Abi</td><td>Kings</td><td>9</td><td>9</td></tr>
<tr><td>...</td><td>...</td><td>...</td><td>...</td></tr>
<tr><td>Kristian</td><td>Fuji</td><td>.</td><td>.</td></tr></table>
```

Import `food_surv_wide.csv` into your `ma505` library as `ma505.food_surv_wide`, and `food_surv.csv` as `ma505.food_surv`.

`guessingrows=max` tells SAS to look at every row before it decides how long each column is. Without it, SAS only looks at the first 20 rows. Kristian is after row 20 in `food_surv.csv`, so his name would get cut off to `Krist`.


In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

proc import datafile="~/food_surv_wide.csv" out=ma505.food_surv_wide dbms=csv replace;
run;

proc import datafile="~/food_surv.csv" out=ma505.food_surv dbms=csv replace;
    guessingrows=max;    /* scan every row, so long names are not cut off */
run;

## How does the DATA step process?

```{=html}
<div class="ds-phases">
  <div class="ds-box comp"><h4>Compilation</h4><ul><li>Checks for syntax errors</li><li>Identifies the column attributes</li><li>Establishes the table metadata</li></ul></div>
  <div class="ds-arrow"><div class="arrow-right black"></div></div>
  <div class="ds-box exec fragment"><h4>Execution</h4><ul><li>Reads and writes data</li><li>Performs data calculations and manipulations</li></ul></div>
</div>
```

::: {.fragment}
During execution, the DATA step:

1. Reads a row from the input table
2. Processes each statement
3. Writes the row to the output table
4. Loops back to the top of the DATA step to read the next row from the input table
:::

## SAS statements

:::: {.columns}
::: {.column width="48%"}
**Declarative**: used in **compilation**

- `LENGTH`, `FORMAT`, `LABEL`, `DROP`, `KEEP`
- Give information to SAS, and only take effect during compilation
- Can be placed in any order
:::
::: {.column width="4%"}
:::
::: {.column width="48%" .fragment}
**Executable**: used in **execution**

- `BMI = Weight / (Height**2) * 703;`
- Anything that tells SAS to do or change something
- Order matters!
:::
::::

## Compilation

During compilation, SAS:

- checks for syntax errors
- creates the **program data vector (PDV)**
- sets up the processing in the PDV
- creates the descriptor (the column information) of the output table

::: {.fragment}
The **PDV** is an area of memory created in compilation. SAS builds **one row at a time** in the PDV.

It always starts with two temporary columns, `_N_` and `_ERROR_`.
:::

## Compilation: building the PDV

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> food_survey;
    <span class="kw">set</span> ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);
<span class="kw">run</span>;</pre></div></div>
```

SAS adds the columns from the input table, then any new columns we create.

```{=html}
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">FoodQuality<span class="pdv-tag">K</span></th><th class="keep">Service<span class="pdv-tag">K</span></th><th class="keep">Score_Sum<span class="pdv-tag">K</span></th><th class="keep">Score_Avg<span class="pdv-tag">K</span></th></tr>
</table></div>
```

::: {.fragment}
`D` means the column is **dropped** (not written to the output table). `K` means it is **kept**. `_N_` counts the iterations, and `_ERROR_` is 1 when there is a data error.
:::

## Execution

During execution, SAS:

1. initializes the PDV (new columns are set to missing)
2. reads a row from the input table into the PDV
3. processes the statements in order, and updates the PDV values
4. at the end of the step, writes the contents of the PDV to the output table
5. returns to the top of the DATA step

::: {.fragment}
Execution only starts if there are no syntax errors!
:::

## Execution: the first row

Click through to see the PDV change after each statement.

```{=html}
<div class="pdv-walk">
<pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> food_survey;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> ma505.food_surv_wide;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    Score_Sum = FoodQuality + Service;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);<span class="fragment pdv-lit" data-fragment-index="4"></span></span><span class="pdv-line"><span class="kw">run</span>;</span></pre>
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">FoodQuality<span class="pdv-tag">K</span></th><th class="keep">Service<span class="pdv-tag">K</span></th><th class="keep">Score_Sum<span class="pdv-tag">K</span></th><th class="keep">Score_Avg<span class="pdv-tag">K</span></th></tr>
<tr class="fragment" data-fragment-index="1"><td>1</td><td>0</td><td></td><td></td><td></td><td></td><td>.</td><td>.</td></tr>
<tr class="fragment" data-fragment-index="2"><td>1</td><td>0</td><td class="hl">Vince</td><td class="hl">Kings</td><td class="hl">7</td><td class="hl">8</td><td>.</td><td>.</td></tr>
<tr class="fragment" data-fragment-index="3"><td>1</td><td>0</td><td>Vince</td><td>Kings</td><td>7</td><td>8</td><td class="hl">15</td><td>.</td></tr>
<tr class="fragment" data-fragment-index="4"><td>1</td><td>0</td><td>Vince</td><td>Kings</td><td>7</td><td>8</td><td>15</td><td class="hl">7.5</td></tr>
</table></div>
</div>
```

::: {.fragment}
At `run;`, SAS writes the row to `work.food_survey`, without `_N_` and `_ERROR_`.
:::

## Execution: the next row

SAS goes back to the top of the DATA step.

```{=html}
<div class="pdv-walk">
<pre class="pdv-code"><span class="pdv-line"><span class="kw">data</span> food_survey;<span class="fragment pdv-lit" data-fragment-index="1"></span></span><span class="pdv-line">    <span class="kw">set</span> ma505.food_surv_wide;<span class="fragment pdv-lit" data-fragment-index="2"></span></span><span class="pdv-line">    Score_Sum = FoodQuality + Service;<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line">    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);<span class="fragment pdv-lit" data-fragment-index="3"></span></span><span class="pdv-line"><span class="kw">run</span>;</span></pre>
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">FoodQuality<span class="pdv-tag">K</span></th><th class="keep">Service<span class="pdv-tag">K</span></th><th class="keep">Score_Sum<span class="pdv-tag">K</span></th><th class="keep">Score_Avg<span class="pdv-tag">K</span></th></tr>
<tr class="fragment" data-fragment-index="1"><td>2</td><td>0</td><td>Vince</td><td>Kings</td><td>7</td><td>8</td><td class="hl">.</td><td class="hl">.</td></tr>
<tr class="fragment" data-fragment-index="2"><td>2</td><td>0</td><td class="hl">Vince</td><td class="hl">Jamaican Grill</td><td class="hl">8</td><td class="hl">6.5</td><td>.</td><td>.</td></tr>
<tr class="fragment" data-fragment-index="3"><td>2</td><td>0</td><td>Vince</td><td>Jamaican Grill</td><td>8</td><td>6.5</td><td class="hl">14.5</td><td class="hl">7.25</td></tr>
</table></div>
</div>
```

::: {.fragment}
At the start of each iteration, columns from the input table **keep** their values until the next row replaces them. **New** columns (`Score_Sum`, `Score_Avg`) are set back to missing.
:::

## When `_ERROR_` is 1

Suppose Abi's service score for Kings was typed as `9D`. That is not a valid number.

```{=html}
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">FoodQuality<span class="pdv-tag">K</span></th><th class="keep">Service<span class="pdv-tag">K</span></th><th class="keep">Score_Sum<span class="pdv-tag">K</span></th><th class="keep">Score_Avg<span class="pdv-tag">K</span></th></tr>
<tr><td>4</td><td class="hl">1</td><td>Abi</td><td>Kings</td><td>9</td><td class="hl">.</td><td>.</td><td>.</td></tr>
</table></div>
```

- SAS sets `_ERROR_` to 1 and writes a note in the log.
- The bad value is set to missing, so `Score_Sum` is missing too.
- SAS keeps going with the next row.

In [ ]:
data food_survey;
    set ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    Score_Avg = mean(FoodQuality, Service);
run;

proc print data=food_survey;
run;

Obs,Student,Restaurant,FoodQuality,Service,Score_Sum,Score_Avg
1,Vince,Kings,7,8,15.0,7.50
2,Vince,Jamaican Grill,8,6.5,14.5,7.25
3,Vince,Fuji,9,6,15.0,7.50
4,Abi,Kings,9,9,18.0,9.00
5,Abi,Jamaican Grill,8,9,17.0,8.50
6,Abi,Fuji,10,9,19.0,9.50
7,Jon,Kings,7,7,14.0,7.00
8,Jon,Jamaican Grill,7,8,15.0,7.50
9,Jon,Fuji,8,6,14.0,7.00
10,Maria,Kings,6,7,13.0,6.50


Look at Kristian. He has no scores, so `Score_Sum` and `Score_Avg` are missing. Notice `FoodQuality + Service` is missing if **either** value is missing, but `mean()` uses whatever values it has.

## Implicit vs explicit output

:::: {.columns}
::: {.column width="48%"}
**Implicit output**

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> food_survey;
    <span class="kw">set</span> ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);
<span class="kw">run</span>;</pre></div></div>
```

- writes the row to the table at the **end** of the iteration
- every DATA step has an implicit output at the end, unless we use an explicit `OUTPUT`
:::
::: {.column width="4%"}
:::
::: {.column width="48%"}
**Explicit output**

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> food_survey;
    <span class="kw">set</span> ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);
    <span class="kw">output</span>;
<span class="kw">run</span>;</pre></div></div>
```

- writes the current row from the PDV **right away**, not at the end
:::
::::

::: {.fragment .red-text}
These two give us the same table.
:::

## But... how would this output look?

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> food_survey2;
    <span class="kw">set</span> ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    <span class="kw">output</span>;
    Score_Avg = <span class="kw">mean</span>(FoodQuality, Service);
    <span class="kw">output</span>;
<span class="kw">run</span>;</pre></div></div>
```

::: {.fragment}
```{=html}
<div class="pdv-wrap">
<div class="pdv-cap">at the first <code>output</code>, then at the second</div>
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">FoodQuality<span class="pdv-tag">K</span></th><th class="keep">Service<span class="pdv-tag">K</span></th><th class="keep">Score_Sum<span class="pdv-tag">K</span></th><th class="keep">Score_Avg<span class="pdv-tag">K</span></th></tr>
<tr><td>1</td><td>0</td><td>Vince</td><td>Kings</td><td>7</td><td>8</td><td class="hl">15</td><td class="hl">.</td></tr>
<tr><td>1</td><td>0</td><td>Vince</td><td>Kings</td><td>7</td><td>8</td><td>15</td><td class="hl">7.5</td></tr>
</table></div>
```
:::

::: {.fragment}
Each row is written **twice**. The first time, `Score_Avg` has not been computed yet, so it is missing.
:::

In [ ]:
data food_survey2;
    set ma505.food_surv_wide;
    Score_Sum = FoodQuality + Service;
    output;
    Score_Avg = mean(FoodQuality, Service);
    output;
run;

proc print data=food_survey2;
run;

Obs,Student,Restaurant,FoodQuality,Service,Score_Sum,Score_Avg
1,Vince,Kings,7,8,15.0,.
2,Vince,Kings,7,8,15.0,7.50
3,Vince,Jamaican Grill,8,6.5,14.5,.
4,Vince,Jamaican Grill,8,6.5,14.5,7.25
5,Vince,Fuji,9,6,15.0,.
6,Vince,Fuji,9,6,15.0,7.50
7,Abi,Kings,9,9,18.0,.
8,Abi,Kings,9,9,18.0,9.00
9,Abi,Jamaican Grill,8,9,17.0,.
10,Abi,Jamaican Grill,8,9,17.0,8.50


## The `RETAIN` statement

Recall: at the start of each iteration, **new** columns are set back to missing.

So what if we want a column `Total` that is the **running total** of `Score`?

::: {.fragment}
If we create `Total` as a new column, it resets to missing on every row, so it can't add up. Instead we use:

```{=html}
<div class="syntax-box"><b>RETAIN</b> <i>column</i> &lt;<i>initial-value</i>&gt;;</div>
```

`RETAIN` leaves the value as it is in the PDV from one row to the next.
:::

## `RETAIN`: in the PDV

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> food;
    <span class="kw">set</span> ma505.food_surv;
    <span class="libref-circle"><span class="kw">retain</span> Total 0;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text"><span class="mono">Total</span> starts at 0, and is not reset on each row</span></span>
    Total = <span class="kw">sum</span>(Total, Score);
<span class="kw">run</span>;</pre></div></div>
```

```{=html}
<div class="pdv-wrap">
<table class="pdv"><tr><th class="drop">_N_<span class="pdv-tag">D</span></th><th class="drop">_ERROR_<span class="pdv-tag">D</span></th><th class="keep">Student<span class="pdv-tag">K</span></th><th class="keep">Restaurant<span class="pdv-tag">K</span></th><th class="keep">Category<span class="pdv-tag">K</span></th><th class="keep">Score<span class="pdv-tag">K</span></th><th class="keep">Total<span class="pdv-tag">K</span></th></tr>
<tr class="fragment"><td>1</td><td>0</td><td></td><td></td><td></td><td>.</td><td class="hl">0</td></tr>
<tr class="fragment"><td>1</td><td>0</td><td class="hl">Vince</td><td class="hl">Kings</td><td class="hl">Food Quality</td><td class="hl">7</td><td>0</td></tr>
<tr class="fragment"><td>1</td><td>0</td><td>Vince</td><td>Kings</td><td>Food Quality</td><td>7</td><td class="hl">7</td></tr>
<tr class="fragment"><td>2</td><td>0</td><td class="hl">Vince</td><td class="hl">Kings</td><td class="hl">Service</td><td class="hl">8</td><td>7</td></tr>
<tr class="fragment"><td>2</td><td>0</td><td>Vince</td><td>Kings</td><td>Service</td><td>8</td><td class="hl">15</td></tr>
</table></div>
```

::: {.fragment}
`sum()` ignores missing values, so `Total` keeps going even when a score is missing.
:::

In [ ]:
data food;
    set ma505.food_surv;
    retain Total 0;
    Total = sum(Total, Score);
run;

proc print data=food;
run;

Obs,Student,Restaurant,Category,Score,Total
1,Vince,Kings,Food Quality,7,7.0
2,Vince,Kings,Service,8,15.0
3,Vince,Jamaican Grill,Food Quality,8,23.0
4,Vince,Jamaican Grill,Service,6.5,29.5
5,Vince,Fuji,Food Quality,9,38.5
6,Vince,Fuji,Service,6,44.5
7,Abi,Kings,Food Quality,9,53.5
8,Abi,Kings,Service,9,62.5
9,Abi,Jamaican Grill,Food Quality,8,70.5
10,Abi,Jamaican Grill,Service,9,79.5
